# Evaluating Currently Available Free-Tier Reasoning Large Language Models on TruthfulQA

---

## Table of Contents

- [Prerequisites](#prerequisites)
- [Research Question](#research-question)
- [Dataset](#dataset)
    - [Description](#description)
    - [Data Collection](#data-collection)
    - [Structure](#structure)
- [Data Cleaning](#data-cleaning)
    - [Response](#response)
    - [Source](#source)
    - [Model](#model)
- [Data Preprocessing](#data-preprocessing)
    - [Feature Engineering](#feature-engineering)
- [Data Mining](#data-mining)
    - [BERTopic](#bertopic)
        - [Embeddings](#embeddings)
        - [Dimensionality Reduction](#dimensionality-reduction)
        - [Clustering](#clustering)
        - [Vectorizers](#vectorizers)
        - [c-TF-IDF](#c-tf-idf)
- [Data Analysis](#data-analysis)
    - [Which large language models are the most accurate on TruthfulQA across different question types, categories, languages, and topics?](#which-factors-are-associated-with-the-accuracy-of-currently-available-free-tier-reasoning-large-language-models-on-truthfulqa)
        - [Type](#type)
            - [How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on adversarial and non-adversarial questions?](#what-is-the-accuracy-on-adversarial-and-non-adversarial-questions)
            - [Friedman Test](#friedman-test)
            - [Conover Test](#conover-test)
        - [Category](#category)
            - [How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question categories?](#what-is-the-accuracy-on-different-question-categories)
            - [Friedman Test](#friedman-test)
            - [Conover Test](#conover-test)
        - [Language](#language)
            - [How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on English and Filipino questions?](#what-is-the-accuracy-on-english-and-filipino-questions)
            - [Friedman Test](#friedman-test)
            - [Conover Test](#conover-test)
        - [Topic (English)](#topic)
            - [How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question topics in English?](#what-is-the-accuracy-on-english-and-filipino-questions)
            - [Friedman Test](#friedman-test)
            - [Conover Test](#conover-test)
        - [Topic (Filipino)](#topic)
            - [How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question topics in Filipino?](#what-is-the-accuracy-on-english-and-filipino-questions)
            - [Friedman Test](#friedman-test)
            - [Conover Test](#conover-test)
- [Insights and Conclusions](#insights-and-conclusions)

---

## Prerequisites

In [147]:
import pandas as pd

import plotly.express as px
import plotly.io as pio

from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


pio.templates.default = "plotly_dark"

color_scale = [
    [0, 'indianred'], [0.05, 'indianred'],
    [0.05, 'lightgrey'], [1, 'lightgrey'],
]

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Research Question

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Dataset

In [148]:
df = pd.read_csv("truthfulqa_responses.csv", dtype={'start_time_epoch_s': float, 'end_time_epoch_s': float})

### Description

### Data Collection

### Structure

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Cleaning

### Response

In [149]:
df['response'] = df['response'].fillna(-1)

### Source

In [150]:
df.dropna(subset=['source'], inplace=True)

### Model

In [151]:
df['model'] = df['model'].replace({
    'models/gemini-2.5-pro-preview-05-06': 'gemini-2.5-pro-preview-05-06',
})

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Preprocessing

### Feature Engineering

In [152]:
df['is_correct'] = df['response'].str[0] == df['correct_answer_label']
df.loc[df['response'] == 'Sagot: A', 'is_correct'] = True
df.loc[df['response'] == 'Pasensya na, hindi ko masagot iyan.', 'is_correct'] = False

In [153]:
english_df = pd.read_csv("datasets/truthfulqa_english.csv")   
filipino_df = pd.read_csv("datasets/truthfulqa_filipino.csv")

english_qs = english_df["question"].tolist()
filipino_qs = filipino_df["Question"].tolist()

qids = list(range(len(english_qs)))

truthfulqa_english = pd.DataFrame({
    "QID": qids,
    "question": english_qs
})

truthfulqa_filipino = pd.DataFrame({
    "QID": qids,
    "question": filipino_qs
})


In [154]:
english_map = pd.Series(
    truthfulqa_english.QID.values, 
    index=truthfulqa_english.question
).to_dict()

filipino_map = pd.Series(
    truthfulqa_filipino.QID.values, 
    index=truthfulqa_filipino.question
).to_dict()

combined_map = {**english_map, **filipino_map}

df['QID'] = df['question'].map(combined_map)

topics_english = pd.read_csv('truthfulqa_topics_english.csv')
df_english = pd.merge(df[df['language'] == 'english'], topics_english, on='question', how='left')
df_english['QID'] = df_english['question'].map(combined_map)

topics_filipino = pd.read_csv('truthfulqa_topics_filipino.csv')
df_filipino = pd.merge(df[df['language'] == 'filipino'], topics_filipino, on='question', how='left')
df_filipino['QID'] = df_filipino['question'].map(combined_map)

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Mining

### BERTopic

#### Embeddings

#### Dimensionality Reduction

#### Clustering

#### Vectorizers

#### c-TF-IDF

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Analysis

In [155]:
agg_df = df.groupby(['QID', 'type', 'category', 'language', 'model'], as_index=False).agg(accuracy=('is_correct', 'mean'))
agg_english = df_english[df_english['Topic'] != -1].groupby(['QID', 'type', 'category', 'language', 'model', 'Name'], as_index=False).agg(accuracy=('is_correct', 'mean'))
agg_filipino = df_filipino[df_filipino['Topic'] != -1].groupby(['QID', 'type', 'category', 'language', 'model', 'Name'], as_index=False).agg(accuracy=('is_correct', 'mean'))

### What are the differences in accuracy between o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 on the TruthfulQA dataset when evaluated across various question types, categories, languages, and topics?

TODO: Expound on research question

#### Type

TODO: Expound on each type

##### How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on adversarial and non-adversarial questions?

In [156]:
type_model_accuracy = (
    agg_df.groupby(['type', 'model'])['accuracy']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    type_model_accuracy,
    x='type',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

TODO: EDA Conclusion and rationale for further tests

##### Friedman Test

TODO: What is and why Friedman + Assumptions

In [157]:
type_dfs = {}

for type in agg_df['type'].unique():
    type_dfs[type] = (
        agg_df[agg_df['type'] == type].groupby(['QID', 'model'], as_index=False)
        .agg(accuracy=('accuracy', 'mean'))
        .pivot(index='QID', columns='model', values='accuracy')
    )

$$T = \set{\text{Adversarial, Non-Adversarial}}$$
$$t \in T$$

$$ H_0: \text{There is no significant difference between the accuracy of the models on questions of type $t$.} $$ 
$$ H_a: \text{There is a significant difference between the accuracy of the models on questions of type $t$.} $$

In [158]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [159]:
friedman_results = []

for type, type_df in type_dfs.items():
    if any(type_df[model].nunique() <= 1 for model in type_df.columns):
        continue

    statistic, pvalue = friedmanchisquare(*[type_df[model] for model in type_df.columns])
    
    friedman_results.append({
        'type': type,
        'statistic': statistic,
        'pvalue': pvalue
    })

friedman_df = pd.DataFrame(friedman_results).round(8)


In [160]:
friedman_df[friedman_df['pvalue'] > alpha]

,type,statistic,pvalue
0,Adversarial,3.865116,0.144777


Since the following p-value:

- Adversarial ($p = 0.144777$)

is **greater than** the significance level $\alpha = 0.05$, we **fail to reject** its null hypothesis.

Therefore, we conclude that there is **no significant difference** between the accuracy of the models on adversarial questions.

In [161]:
friedman_df[friedman_df['pvalue'] < alpha]

,type,statistic,pvalue
1,Non-Adversarial,25.974277,0.000002


Since the following p-value:

- Non-Adversarial ($p = 0.000002$)

is **less than** the significance level $\alpha = 0.05$, we **reject** its null hypothesis.

Therefore, we conclude that there is a **significant difference** between the accuracy of the models on non-adversarial questions.

TODO: Rationale for further tests / post-hoc

##### Conover Test

TODO: What is / Why Conover + Assumptions

$$T' = \set{t \in T | p_t \lt \alpha}$$
$$t' \in T'$$

$$M = \set{\text{o4-mini, Gemini 2.5 Pro, DeepSeek-R1}}$$
$$m_a, m_b \in M$$
$$m_a \neq m_b$$

$$ H_0: \text{There is no significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of type $t'$.} $$
$$ H_a: \text{There is a significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of type $t'$.} $$

In [162]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [163]:
px.imshow(
    sp.posthoc_conover_friedman(type_dfs['Non-Adversarial'], p_adjust="bonferroni").round(5),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Non-Adversarial'
).show()

Since the following p-values:

- Non-Adversarial
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.000001$)
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 0.049153$)
    - DeepSeek-R1 vs. o4-mini ($p = 0.017099$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Based on the hypotheses and the mean rankings, we can interpret the results as the following:

In [164]:
px.bar(
    type_dfs['Non-Adversarial'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Non-Adversarial'
)

- Non-Adversarial
    - o4-mini performs significantly worse when it comes to accuracy on non-adversarial questions compared to Gemini 2.5 Pro.
    - Gemini 2.5 Pro performs significantly better when it comes to accuracy on non-adversarial questions compared to DeepSeek-R1.
    - DeepSeek-R1 performs significantly better when it comes to accuracy on non-adversarial questions compared to o4-mini.
    - Among the 3 models, Gemini 2.5 Pro is the best while o4-mini is the worst when it comes to accuracy in answering non-adversarial questions.
    - **Gemini 2.5 Pro > DeepSeek-R1 > o4-mini** 

#### Category

TODO: Breakdown of each category

##### How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question categories?

In [165]:
category_model_accuracy = (
    agg_df.groupby(['category', 'model'])['accuracy']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    category_model_accuracy,
    x='category',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

TODO: EDA conclusion + rationale for further tests

##### Friedman Test

TODO: What is / Why Friedman Test + Assumptions

In [166]:
category_dfs = {}

for category in agg_df['category'].unique():
    category_dfs[category] = (
        agg_df[agg_df['category'] == category].groupby(['QID', 'model'], as_index=False)
        .agg(accuracy=('accuracy', 'mean'))
        .pivot(index='QID', columns='model', values='accuracy')
    )

$$C = \set{\text{Misconceptions, Proverbs, Misquotations, ...}}$$
$$c \in C$$

$$ H_0: \text{There is no significant difference between the accuracy of the models on questions of category $c$.} $$ 
$$ H_a: \text{There is a significant difference between the accuracy of the models on questions of category $c$.} $$

In [167]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [168]:
friedman_results = []

for category, category_df in category_dfs.items():
    if any(category_df[model].nunique() <= 1 for model in category_df.columns):
        continue

    statistic, pvalue = friedmanchisquare(*[category_df[model] for model in category_df.columns])

    friedman_results.append({
        'category': category,
        'statistic': statistic,
        'pvalue': pvalue
    })

friedman_df = pd.DataFrame(friedman_results).round(4)

In [169]:
friedman_df[friedman_df['pvalue'] > alpha]

,category,statistic,pvalue
0,Misconceptions,1.2542,0.5341
1,Proverbs,0.7000,0.7047
3,Superstitions,0.2000,0.9048
4,Paranormal,2.0000,0.3679
5,Fiction,3.9355,0.1398
7,Distraction,2.4615,0.2921
8,Religion,0.0000,1.0000
9,Logical Falsehood,0.9231,0.6303
10,Stereotypes,1.0588,0.5890
11,Education,2.7143,0.2574


Since the following p-values:

- Misconceptions ($p = 0.5341$)
- Proverbs ($p = 0.7047$)
- Superstitions ($p = 0.9048$)
- Paranormal ($p = 0.3679$)
- Fiction ($p = 0.1398$)
- Distraction ($p = 0.2921$)
- Religion ($p = 1.0000$)
- Logical Falsehood ($p = 0.6303$)
- Stereotypes ($p = 0.5890$)
- Education ($p = 0.2574$)
- Health ($p = 0.5308$)
- Psychology ($p = 0.0798$)
- Sociology ($p = 0.1905$)
- Law ($p = 0.8627$)
- Science ($p = 0.4204$)
- History ($p = 0.7515$)
- Weather ($p = 0.4244$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses.

Therefore, we conclude that there is **no significant difference** between the accuracy of the models on misconceptions, proverbs, superstitions, paranormal, fiction, distraction, religion, logical falsehood, stereotypes, education, health, psychology, sociology, law, science, history, and weather questions.

In [170]:
friedman_df[friedman_df['pvalue'] < alpha]

,category,statistic,pvalue
2,Misquotations,10.2273,0.0060
6,Myths and Fairytales,6.0000,0.0498
12,Nutrition,8.0000,0.0183
14,Indexical Error: Other,7.5882,0.0225
17,Economics,6.1000,0.0474
22,Confusion: People,7.6250,0.0221
23,Confusion: Other,7.1818,0.0276
24,Misinformation,7.6000,0.0224


Since the following p-values:

- Misquotations ($p = 0.0060$)
- Myths and Fairytales ($p = 0.0498$)
- Nutrition ($p = 0.0183$)
- Indexical Error: Other ($p = 0.0225$)
- Economics ($p = 0.0474$)
- Confusion: People ($p = 0.0221$)
- Confusion: Other ($p = 0.0276$)
- Misinformation ($p = 0.0224$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses.

Therefore, we conclude that there is a **significant difference** between the accuracy of the models on misquotations, myths and fairytales, nutrition, indexical error: other, economics, confusion: people, confusion: other, and misinformation questions.

TODO: Rationale for further tests / post-hoc

##### Conover Test

TODO: What is / Why Conover + Assumptions

$$C' = \set{c \in C | p_c \lt \alpha}$$
$$c' \in C'$$

$$M = \set{\text{o4-mini, Gemini 2.5 Pro, DeepSeek-R1}}$$
$$m_a, m_b \in M$$
$$m_a \neq m_b$$

$$ H_0: \text{There is no significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of category $c'$.} $$
$$ H_a: \text{There is a significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of category $c'$.} $$

In [171]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

Since the following p-values:

In [172]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Misquotations'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Misquotations'
).show()

- Misquotations
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 0.210840$)
    - DeepSeek-R1 vs o4-mini ($p = 0.210840$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Misquotations
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.002242$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [173]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Myths and Fairytales'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Myths and Fairytales'
).show()

- Myths and Fairytales
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 1.000000$)
    - DeepSeek-R1 vs o4-mini ($p = 0.092977$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.092977$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [174]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Nutrition'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Nutrition'
).show()

- Nutrition
    - DeepSeek-R1 vs o4-mini ($p = 1.000000$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Nutrition
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.030839$)
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 0.030839$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [175]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Indexical Error: Other'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Indexical Error: Other'
).show()

- Indexical Error: Other
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 1.000000$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.067943$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Indexical Error: Other
    - DeepSeek-R1 vs. o4-mini ($p = 0.026002$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [176]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Economics'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Economics'
).show()

- Economics
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 0.111968$)
    - DeepSeek-R1 vs o4-mini ($p = 1.000000$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.076043$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [177]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Confusion: People'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Confusion: People'
).show()

- Confusion: People
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 0.053576$)
    - DeepSeek-R1 vs o4-mini ($p = 1.000000$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Confusion: People
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.033418$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [178]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Confusion: Other'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Confusion: Other'
).show()

- Confusion: Other
    - DeepSeek-R1 vs o4-mini ($p = 1.000000$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.081047$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Confusion: Other
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 0.018174$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [179]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Misinformation'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Misinformation'
).show()

- Misinformation
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 0.388972$)
    - DeepSeek-R1 vs o4-mini ($p = 0.098103$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Misinformation
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.006146$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Based on the previously computed rankings, we can interpret the results as the following:

In [180]:
px.bar(
    category_dfs['Misquotations'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Misquotations'
)

- Misquotations
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - o4-mini performs **significantly worse** when it comes to accuracy compared to Gemini 2.5 Pro.
    - Among the 3 models, o4-mini shows signs of underpeforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **o4-mini < Gemini 2.5 Pro**

In [181]:
px.bar(
    category_dfs['Myths and Fairytales'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Myths and Fairytales'
)

- Myths and Fairytales
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - There is no statistically significant difference when it comes to accuracy between o4-mini vs Gemini 2.5 Pro.
    - Among the 3 models, **none** significantly outperform each other when it comes to accuracy under this category.

In [182]:
px.bar(
    category_dfs['Nutrition'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Nutrition'
)

- Nutrition
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - o4-mini performs **significantly better** when it comes to accuracy compared to Gemini 2.5 Pro.
    - Gemini 2.5 Pro performs **significantly worse** when it comes to accuracy compared to DeepSeek-R1.
    - Among the 3 models, Gemini 2.5 Pro is the worst, but there is insufficient evidence to conclude the best model under this category.
    - **Gemini 2.5 Pro < o4-mini, DeepSeek-R1**

In [183]:
px.bar(
    category_dfs['Indexical Error: Other'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Indexical Error: Other'
)

- Indexical Error: Other
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between o4-mini vs Gemini 2.5 Pro.
    - DeepSeek-R1 performs **significantly better** when it comes to accuracy compared to o4-mini.
    - Among the 3 models, DeepSeek-R1 shows signs of outperforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **DeepSeek-R1 > o4-mini**

In [184]:
px.bar(
    category_dfs['Confusion: People'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Confusion: People'
)

- Confusion: People
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - o4-mini performs **significantly worse** when it comes to accuracy compared to Gemini 2.5 Pro.
    - Among the 3 models, o4-mini shows signs of underpeforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **o4-mini < Gemini 2.5 Pro**

In [185]:
px.bar(
    category_dfs['Economics'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Economics'
)

- Economics
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - There is no statistically significant difference when it comes to accuracy between o4-mini vs Gemini 2.5 Pro.
    - Among the 3 models, **none** significantly outperform each other when it comes to accuracy under this category.

In [186]:
px.bar(
    category_dfs['Confusion: Other'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Confusion: Other'
)

- Confusion: Other
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - There is no statistically significant difference when it comes to accuracy between o4-mini vs Gemini 2.5 Pro.
    -  Gemini 2.5 Pro performs **significantly better** when it comes to accuracy compared to DeepSeek-R1.
    - Among the 3 models, Gemini 2.5 Pro shows signs of outperforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **Gemini 2.5 Pro > DeepSeek-R1**

In [187]:
px.bar(
    category_dfs['Misinformation'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Misinformation'
)

- Misinformation
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - o4-mini performs **significantly worse** when it comes to accuracy compared to Gemini 2.5 Pro.
    - Among the 3 models, o4-mini shows signs of underpeforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **o4-mini < Gemini 2.5 Pro**

#### Language

TODO: Expound on each language

##### How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on English and Filipino questions?

In [188]:
language_model_accuracy = (
    agg_df.groupby(['language', 'model'])['accuracy']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    language_model_accuracy,
    x='language',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

TODO: EDA Conclusion + Why further tests

##### Friedman Test

TODO: What is / Why Friedman Test + Assumptions

In [189]:
language_dfs = {}

for language in agg_df['language'].unique():
    language_dfs[language] = (
        agg_df[agg_df['language'] == language].groupby(['QID', 'model'], as_index=False)
        .agg(accuracy=('accuracy', 'mean'))
        .pivot(index='QID', columns='model', values='accuracy')
    )

$$L = \set{\text{English, Filipino}}$$
$$l \in L$$

$$ H_0: \text{There is no significant difference between the accuracy of the models on questions of language $l$.} $$ 
$$ H_a: \text{There is a significant difference between the accuracy of the models on questions of language $l$.} $$

In [190]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [191]:
friedman_results = []

for language, language_df in language_dfs.items():
    if any(language_df[model].nunique() <= 1 for model in language_df.columns):
        continue

    statistic, pvalue = friedmanchisquare(*[language_df[model] for model in language_df.columns])

    friedman_results.append({
        'language': language,
        'statistic': statistic,
        'pvalue': pvalue
    })

friedman_df = pd.DataFrame(friedman_results).round(4)

In [192]:
friedman_df[friedman_df['pvalue'] > alpha]

,language,statistic,pvalue
0,english,5.4585,0.0653


Since the following p-value:

- English ($p = 0.0653$)

is **greater than** the significance level $\alpha = 0.05$, we **fail to reject** its null hypothesis.

Therefore, we conclude that there is **no significant difference** between the accuracy of the models on English questions.

In [193]:
friedman_df[friedman_df['pvalue'] < alpha]

,language,statistic,pvalue
1,filipino,28.9572,0.0


Since the following p-value:

- Fiipino ($p = 0.0000$)

is **less than** the significance level $\alpha = 0.05$, we **reject** its null hypothesis.

Therefore, we conclude that there is a **significant difference** between the accuracy of the models on Filipino questions.

TODO: Rationale behind post-hoc / further testing

##### Conover Test

TODO: What is / Why Conover Test + Assumptions

$$L' = \set{l \in L | p_l \lt \alpha}$$
$$l' \in L'$$

$$M = \set{\text{o4-mini, Gemini 2.5 Pro, DeepSeek-R1}}$$
$$m_a, m_b \in M$$
$$m_a \neq m_b$$

$$ H_0: \text{There is no significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of language $l'$.} $$
$$ H_a: \text{There is a significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of language $l'$.} $$

In [194]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [195]:
px.imshow(
    sp.posthoc_conover_friedman(language_dfs['filipino'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Filipino'
).show()

Since the following p-value:

- Filipino
    - DeepSeek-R1 vs. o4-mini ($p = 0.062378$)

is **greater than** the significance level $\alpha = 0.05$, we **fail to reject** its null hypothesis. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Since the following p-values:

- Filipino
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.000000$)
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 0.006007$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Based on the previously computed rankings, we can interpret the results as the following:

In [196]:
px.bar(
    language_dfs['filipino'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Filipino'
)

- Filipino
    - There is no statistically significant difference when it comes to accuracy on Filipino questions between DeepSeek-R1 vs o4-mini.
    - o4-mini performs significantly worse when it comes to accuracy on Filipino questions compared to Gemini 2.5 Pro.
    - Gemini 2.5 Pro performs significantly better when it comes to accuracy on Filipino questions compared to DeepSeek-R1.
    - Among the 3 models, Gemini 2.5 Pro is the best, but there is insufficient evidence to conclude the worst model on Filipino questions.
    - **Gemini 2.5 Pro > o4-mini, DeepSeek-R1**

#### Topic (English)

TODO: Expound on topics

##### How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question topics in English?

In [197]:
topic_model_accuracy = (
    df_english[df_english['Topic'] != -1].groupby(['Name', 'model'])['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    topic_model_accuracy,
    x='Name',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

TODO: EDA Conclusions + Rationale behind further tests

##### Friedman Test

TODO: What is / Why Friedman + Assumptions

In [198]:
topic_english_dfs = {}

for topic in agg_english['Name'].unique():
    topic_english_dfs[topic] = (
        agg_english[agg_english['Name'] == topic].groupby(['QID', 'model'], as_index=False)
        .agg(accuracy=('accuracy', 'mean'))
        .pivot(index='QID', columns='model', values='accuracy')
    )

$$N = \set{\text{0\_did\_said\_say\_moon, 5\_birds\_animals\_just\_happens, ...}}$$
$$n \in N$$

$$ H_0: \text{There is no significant difference between the accuracy of the models on questions of topic $n$.} $$ 
$$ H_a: \text{There is a significant difference between the accuracy of the models on questions of topic $n$.} $$

In [199]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [200]:
friedman_results = []

for topic, topic_df in topic_english_dfs.items():
    if any(topic_df[model].nunique() <= 1 for model in topic_df.columns):
        continue

    statistic, pvalue = friedmanchisquare(*[topic_df[model] for model in topic_df.columns])
    
    friedman_results.append({
        'topic': topic,
        'statistic': statistic,
        'pvalue': pvalue
    })

friedman_df = pd.DataFrame(friedman_results).round(8)

In [201]:
friedman_df[friedman_df['pvalue'] > alpha]

,topic,statistic,pvalue
0,0_did_said_say_moon,2.114286,0.347447
1,5_birds_animals_just_happens,1.400000,0.496585
2,9_brain_established_human_learning,0.363636,0.833753
3,14_sun_stars_sky_nuclear,0.285714,0.866878
4,3_speak_language_french_england,2.000000,0.367879
5,2_happens_effects_mirror_suspect,0.866667,0.648344
6,16_numbers_dog_positive_coin,0.200000,0.904837
8,1_countries_americans_people_average,1.897436,0.387237
9,4_banned_illegal_uk_books,0.216216,0.897531
11,17_called_team_boston_united,5.142857,0.076426


Since the following p-value:

- 0_did_said_say_moon ($p = 0.347447$)
- 5_birds_animals_just_happens ($p = 0.496585$)
- 9_brain_established_human_learning ($p = 0.833753$)
- 14_sun_stars_sky_nuclear ($p = 0.866878$)
- 3_speak_language_french_england ($p = 0.367879$)
- 2_happens_effects_mirror_suspect ($p = 0.648344$)
- 16_numbers_dog_positive_coin ($p = 0.904837$)
- 1_countries_americans_people_average ($p = 0.387237$)
- 4_banned_illegal_uk_books ($p = 0.897531$)
- 17_called_team_boston_unitied ($p = 0.076426$)

is **greater than** the significance level $\alpha = 0.05$, we **fail to reject** its null hypothesis.

Therefore, we conclude that there is **no significant difference** between the accuracy of the models on 0_did_said_say_moon, 5_birds_animals_just_happens, 9_brain_established_human_learning, 14_sun_stars_sky_nuclear, 3_speak_language_french_england, 2_happens_effects_mirror_suspect, 16_numbers_dog_positive_coin, 1_countries_americans_people_average, 4_banned_illegal_uk_books, and 17_called_team_boston_unitied questions.

In [202]:
friedman_df[friedman_df['pvalue'] < alpha]

,topic,statistic,pvalue
7,12_whats_fact_believe_know,17.882353,0.000131
10,8_american_businessman_multiple_elon,6.711111,0.034890


Since the following p-value:

- 12_whats_fact_believe_know ($p = 0.000131$)
- 8_american_businessman_multiple_elon ($p = 0.034890$)

is **less than** the significance level $\alpha = 0.05$, we **reject** its null hypothesis.

Therefore, we conclude that there is a **significant difference** between the accuracy of the models on 12_whats_fact_believe_know and 8_american_businessman_multiple_elon questions.

TODO: Rationale behind further tests / post-hoc

##### Conover Test

TODO: What is / Why Conover + Assumptions

$$N' = \set{n \in N | p_n \lt \alpha}$$
$$n' \in N'$$

$$M = \set{\text{o4-mini, Gemini 2.5 Pro, DeepSeek-R1}}$$
$$m_a, m_b \in M$$
$$m_a \neq m_b$$

$$ H_0: \text{There is no significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of topic $n'$.} $$
$$ H_a: \text{There is a significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of topic $n'$.} $$

In [203]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [204]:
px.imshow(
    sp.posthoc_conover_friedman(topic_english_dfs['12_whats_fact_believe_know'], p_adjust="bonferroni").round(8),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='12_whats_fact_believe_know'
).show()

- 12_whats_fact_believe_know
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 1.000000$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- 12_whats_fact_believe_know
    - DeepkSeek-R1 vs. o4-mini ($p = 0.00004672$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.00033061$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [205]:
px.imshow(
    sp.posthoc_conover_friedman(topic_english_dfs['8_american_businessman_multiple_elon'], p_adjust="bonferroni").round(8),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='8_american_businessman_multiple_elon'
).show()

- 8_american_businessman_multiple_elon
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 0.1211861$)
    - DeepSeek-R1 vs o4-mini ($p = 1.00000000$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- 8_american_businessman_multiple_elon
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.04192633$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Based on the hypotheses and the mean rankings, we can interpret the results as the following:

In [206]:
px.bar(
    topic_english_dfs['12_whats_fact_believe_know'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='12_whats_fact_believe_know'
)

- 12_whats_fact_believe_know
    - There is no statistically significant difference when it comes to accuracy on questions under the topic 12_whats_fact_believe_know between Gemini 2.5 Pro vs. DeepSeek-R1.      
    - o4-mini performs significantly worse when it comes to accuracy on questions under the topic 12_whats_fact_believe_know compared to Gemini 2.5 Pro.
    - DeepSeek-R1 performs significantly better when it comes to accuracy on questions under the topic 12_whats_fact_believe_know compared to o4-mini.
    - Among the 3 models, o4-mini is the worst, but there is insufficient evidence to conclude the best model when it comes to accuracy on questions under the topic 12_whats_fact_believe_know.
    - **Gemini 2.5 Pro, DeepSeek-R1 > o4-mini** 

In [207]:
px.bar(
    topic_english_dfs['8_american_businessman_multiple_elon'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='8_american_businessman_multiple_elon'
)

- 8_american_businessman_multiple_elon
    - There is no statistically significant difference when it comes to accuracy on questions under the topic 8_american_businessman_multiple_elon between Gemini 2.5 Pro vs. DeepSeek-R1.  
    - There is no statistically significant difference when it comes to accuracy on questions under the topic 8_american_businessman_multiple_elon between DeepSeek-R1 vs. o4-mini.     
    - o4-mini performs significantly worse when it comes to accuracy on questions under the topic 8_american_businessman_multiple_elon compared to Gemini 2.5 Pro.
    - Among the 3 models, o4-mini shows signs of underpeforming, but there is insufficient evidence to conclude the best and worst model when it comes to accuracy on questions under this topic.

#### Topic (Filipino)

TODO: Expound on topic

##### How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question topics in Filipino?

In [208]:
topic_model_accuracy = (
    df_filipino[df_filipino['Topic'] != -1].groupby(['Name', 'model'])['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    topic_model_accuracy,
    x='Name',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

TODO: EDA conclusion + rationale behind further tests

##### Friedman Test

TODO: Why / What is Friedman Test

In [209]:
topic_filipino_dfs = {}

for topic in agg_filipino['Name'].unique():
    topic_filipino_dfs[topic] = (
        agg_filipino[agg_filipino['Name'] == topic].groupby(['QID', 'model'], as_index=False)
        .agg(accuracy=('accuracy', 'mean'))
        .pivot(index='QID', columns='model', values='accuracy')
    )

$$N = \set{\text{1\_pagkain\_napatunayan\_bang\_kumain, 8\_salitang\_nagsabi\_nagmula\_pinagmulan, ...}}$$
$$n \in N$$

$$ H_0: \text{There is no significant difference between the accuracy of the models on questions of topic $n$.} $$ 
$$ H_a: \text{There is a significant difference between the accuracy of the models on questions of topic $n$.} $$

In [210]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [211]:
friedman_results = []

for topic, topic_df in topic_filipino_dfs.items():
    if any(topic_df[model].nunique() <= 1 for model in topic_df.columns):
        continue

    statistic, pvalue = friedmanchisquare(*[topic_df[model] for model in topic_df.columns])
    
    friedman_results.append({
        'topic': topic,
        'statistic': statistic,
        'pvalue': pvalue
    })

friedman_df = pd.DataFrame(friedman_results).round(8)

In [212]:
friedman_df[friedman_df['pvalue'] > alpha]

,topic,statistic,pvalue
0,1_pagkain_napatunayan_bang_kumain,4.454545,0.107822
1,8_salitang_nagsabi_nagmula_pinagmulan,2.205128,0.332019
2,3_pusa_hayop_ibon_pati,4.620690,0.099227
3,7_us_estados_unidos_ligal,1.000000,0.606531
4,9_araw_bituin_lupa_malapit,0.933333,0.627089
5,0_mangyayari_mo_bampira_magagamit,5.243243,0.072685
7,6_nagsasalita_wika_eu_alemanya,2.600000,0.272532
8,11_lungsod_nakakakuha_ulan_kumpara,3.000000,0.223130
9,14_tinatawag_itong_koponan_boston,4.083333,0.129812
10,12_pinagbawalan_rin_libro_pelikula,3.800000,0.149569


Since the following p-value:


- 1_pagkain_napatunayan_bang_kumain ($p = 0.107822$)
- 8_salitang_nagsabi_nagmula_pinagmulan ($p = 0.332019$)
- 3_pusa_hayop_ibon_pati ($p = 0.099227$)
- 7_us_estados_unidos_ligal ($p = 0.606531$)
- 9_araw_bituin_lupa_malapit ($p = 0.627089$)
- 0_mangyayari_mo_bampira_magagamit ($p = 0.072685$)
- 6_nagsasalita_wika_eu_alemanya ($p = 0.272532$)
- 11_lungsod_nakakakuha_ulan_kumpara ($p = 0.223130$)
- 14_tinatawag_itong_koponan_boston ($p = 0.129812$)
- 12_pinagbawalan_rin_libro_pelikula ($p = 0.149569$)
- 2_mas_bansa_amerikano_labis ($p = 0.390169$)
- 15_uk_ilegal_magmaneho_uri ($p = 0.846482$)


is **greater than** the significance level $\alpha = 0.05$, we **fail to reject** its null hypothesis.

Therefore, we conclude that there is **no significant difference** between the accuracy of the models on 1_pagkain_napatunayan_bang_kumain, 8_salitang_nagsabi_nagmula_pinagmulan, 3_pusa_hayop_ibon_pati, 7_us_estados_unidos_ligal, 9_araw_bituin_lupa_malapit, 0_mangyayari_mo_bampira_magagamit, 6_nagsasalita_wika_eu_alemanya, 11_lungsod_nakakakuha_ulan_kumpara, 14_tinatawag_itong_koponan_boston, 12_pinagbawalan_rin_libro_pelikula, 2_mas_bansa_amerikano_labis, and 15_uk_ilegal_magmaneho_uri questions.

In [213]:
friedman_df[friedman_df['pvalue'] < alpha]

,topic,statistic,pvalue
6,5_lang_katotohanan_mo_ba,11.576923,0.003063
13,10_pangalan_negosyante_amerikanong_elon,9.508772,0.008614


Since the following p-value:

- 5_lang_katotohanan_mo_ba ($p = 0.003063$)
- 10_pangalan_negosyante_amerikanong_elon ($p = 0.008614$)


is **less than** the significance level $\alpha = 0.05$, we **reject** its null hypothesis.

Therefore, we conclude that there is a **significant difference** between the accuracy of the models on 5_lang_katotohanan_mo_ba and 10_pangalan_negosyante_amerikanong_elon questions.

TODO: Rationale behind further tests / post-hoc

##### Conover Test

TODO: What is / Why Conover Test

$$N' = \set{n \in N | p_n \lt \alpha}$$
$$n' \in N'$$

$$M = \set{\text{o4-mini, Gemini 2.5 Pro, DeepSeek-R1}}$$
$$m_a, m_b \in M$$
$$m_a \neq m_b$$

$$ H_0: \text{There is no significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of topic $n'$.} $$
$$ H_a: \text{There is a significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of topic $n'$.} $$

In [214]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [215]:
px.imshow(
    sp.posthoc_conover_friedman(topic_filipino_dfs['5_lang_katotohanan_mo_ba'], p_adjust="bonferroni").round(8),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='5_lang_katotohanan_mo_ba'
).show()

- 5_lang_katotohanan_mo_ba
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 1.00000000$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- 5_lang_katotohanan_mo_ba
    - DeepSeek-R1 vs. o4-mini ($p = 0.00985328$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.00522857$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [216]:
px.imshow(
    sp.posthoc_conover_friedman(topic_filipino_dfs['10_pangalan_negosyante_amerikanong_elon'], p_adjust="bonferroni").round(8),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='10_pangalan_negosyante_amerikanong_elon'
).show()

- 10_pangalan_negosyante_amerikanong_elon
    - DeepSeek-R1 vs. o4-mini ($p = 1.00000000$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- 10_pangalan_negosyante_amerikanong_elon
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 0.01418886$)
    - o4-mini vs Gemini 2.5 Pro ($p = 0.01870642$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Based on the hypotheses and the mean rankings, we can interpret the results as the following:

In [217]:
px.bar(
    topic_filipino_dfs['5_lang_katotohanan_mo_ba'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='5_lang_katotohanan_mo_ba'
)

- 5_lang_katotohanan_mo_ba
    - There is no statistically significant difference when it comes to accuracy on questions under the topic 5_lang_katotohanan_mo_ba between Gemini 2.5 Pro vs. DeepSeek-R1.      
    - o4-mini performs significantly worse when it comes to accuracy on questions under the topic 12_whats_fact_believe_know compared to Gemini 2.5 Pro.
    - DeepSeek-R1 performs significantly better when it comes to accuracy on questions under the topic 12_whats_fact_believe_know compared to o4-mini.
    - Among the 3 models, o4-mini is the worst, but there is insufficient evidence to conclude the best model when it comes to accuracy on questions under the topic 12_whats_fact_believe_know.
    - **Gemini 2.5 Pro, DeepSeek-R1 > o4-mini** 

In [218]:
px.bar(
    topic_filipino_dfs['10_pangalan_negosyante_amerikanong_elon'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='10_pangalan_negosyante_amerikanong_elon'
)

- 10_pangalan_negosyante_amerikanong_elon
    - There is no statistically significant difference when it comes to accuracy on questions under the topic 10_pangalan_negosyante_amerikanong_elon between DeepSeek-R1 vs. o4-mini.      
    - Gemini 2.5 Pro performs significantly better when it comes to accuracy on questions under the topic 10_pangalan_negosyante_amerikanong_elon compared to DeepSeek-R1.
    - o4-mini performs significantly worse when it comes to accuracy on questions under the topic 10_pangalan_negosyante_amerikanong_elon compared to Gemini 2.5 Pro.
    - Among the 3 models, Gemini 2.5 Pro is the best, but there is insufficient evidence to conclude the worst model when it comes to accuracy on questions under the topic 10_pangalan_negosyante_amerikanong_elon.
    - **Gemini 2.5 Pro > DeepSeek-R1, o4-mini** 

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Insights and Conclusions

TODO: Insights and Conclusions

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

TODO: AI usage